# Advanced document indexing

# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

In [1]:
# Run this cell in Google Colab before running the rest of the notebook.
%pip install -q langchain==1.0.3 langchain-openai==1.0.1 langchain-community==0.4.1 langchain-chroma==1.0.0 langchain-openrouter chromadb==1.3.0 lxml==5.4.0 html2text==2025.4.15 lark==1.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 M

### Preparing the Chroma DB collections

In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import os
import getpass

try:
    from google.colab import userdata
except ImportError:
    userdata = None

OPENROUTER_API_KEY = None
if userdata is not None:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OPENROUTER_API_KEY: ")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ.setdefault("USER_AGENT", "building-llm-applications/ch08-colab")

embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=embedding_model,
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists

In [ ]:
corwnall_coarse_collection = Chroma( #A
    collection_name="cornwall_coarse",
    embedding_function=embedding_model,
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists

### Loading the HTML content with the AsyncHtmlLoader

In [ ]:
from langchain_community.document_loaders import AsyncHtmlLoader

In [ ]:
# destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
destination_url = "https://en.wikivoyage.org/api/rest_v1/page/html/Cornwall"

In [ ]:
# html_loader = AsyncHtmlLoader(destination_url)


In [ ]:
# docs = html_loader.load()

In [17]:
import requests
from urllib.parse import quote
from langchain_core.documents import Document

page_title = "Cornwall"

url = f"https://en.wikivoyage.org/api/rest_v1/page/html/{quote(page_title)}"

headers = {
    "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
    "Accept-Encoding": "gzip",
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

docs = [
    Document(
        page_content=response.text,
        metadata={"source": url, "title": page_title},
    )
]

In [ ]:
# from urllib.parse import quote
# from langchain_core.documents import Document

# page_title = "Cornwall"

# url = f"https://en.wikivoyage.org/api/rest_v1/page/html/{quote(page_title)}"

# headers = {
#     "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
#     "Accept-Encoding": "gzip",
# }

# html_loader = AsyncHtmlLoader(
#     url,
#     header_template=headers,
#     requests_per_second=1,
#     raise_for_status=True,
# )

# docs = html_loader.load()
# for doc in docs:
#     doc.metadata.update({"source": url, "title": page_title})

In [ ]:
len(docs)

1

### Splitting into granular chunks with the HTMLSectionSplitter

In [ ]:
from langchain_text_splitters import HTMLSectionSplitter

In [ ]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [ ]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks)

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [ ]:
granular_chunks = split_docs_into_granular_chunks(docs)

#### Ingesting granular chunks

In [ ]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['93e42006-9014-4bae-b818-2541998aa014',
 '9d9265ea-182c-458d-8379-86010c82b7f6',
 '048024e9-1775-4cc3-af37-7824110f709f',
 'f08feec8-7084-468e-8d94-b0c578a86ce9',
 '0a18cfae-be33-4c7e-8f13-5139769ade24',
 '1f7b438d-4471-4a6e-b9c3-4c9910c941c7',
 '50b3b4d9-d1b8-497d-8946-0a7a17b618ab',
 '0eb34d3c-55dc-43b1-b80f-9c1f99876c7b',
 '4ecb2835-3ff2-4397-b549-b6c4451bfaa8',
 '9248bb98-10cb-4d2b-af01-448c0b2c0dd1',
 '54139460-f319-4c82-9fda-34e7eac6d487',
 'e348fb58-5678-45fb-943b-10f8b25bd358',
 'da7771fc-8e32-4cd2-8b21-5cef0c69d2d2',
 '61b8e33b-ed4d-4376-a0c1-1039b23456ec',
 '7a6e76e4-6ac0-4a7f-8582-447476d65cea']

#### Searching granular chunks

In [ ]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='Festivals 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade during this peri

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [12]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [13]:
html2text_transformer = Html2TextTransformer()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [ ]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [ ]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [ ]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['7882874c-2f2e-43d4-8ae6-166846a000fb',
 '93f66a45-a0a8-47d0-9c12-acc7972c6707',
 'bb3d7c29-5bbc-44e6-98c1-e3772c5b486c',
 'a2edfdf6-6568-41ef-aa05-30e69d9732b0',
 '550b3c29-81c7-4124-9446-8e9e531649b8',
 '8d879113-be1b-466d-84f6-e016c2b24605',
 '8b1690be-8431-4965-a52a-cb1b3796af79',
 'cfdc638b-f1b3-43f0-8aab-f327af364a05',
 '17b181cf-7c9f-4fcc-b548-cc7d258e93f5',
 '2b7114ba-122a-4b4e-a052-712452c87dcb',
 '112f72f5-5d64-48ca-b412-98553285f961',
 'be8d3461-c7e7-40de-b924-6670cd12bd72',
 '20a48c30-440a-42b4-b826-204bd89d2ed2']

#### Searching coarse chunks

In [ ]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

page_content='_See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Cornish fishing communities. The celeb

## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [ ]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=embedding_model,
)

uk_granular_collection.reset_collection() #B

In [ ]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=embedding_model,
)

uk_coarse_collection.reset_collection() #B

### Splitting and ingesting HTML content with the HTMLSectionSplitter

In [3]:
# Reduce this list if you want to save on processing fees
# uk_destinations = [
#     "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
#     "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
#     "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
#     "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
#     "Rye_(England)", "Seaford", "Ashdown_Forest"
# ]

uk_destinations = [
     "Cornwall", "East_Sussex", "Polperro", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

wikivoyage_headers = {
    "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
    "Accept-Encoding": "gzip",
}

def load_wikivoyage_html(url):
    try:
        response = requests.get(url, headers=wikivoyage_headers, timeout=30)
        response.raise_for_status()
        return [Document(page_content=response.text, metadata={"source": url})]
    except Exception as e:
        print(f"Skipping {url}: {e}")
        return []

In [ ]:
# import asyncio
# uk_destinations = [
#      "Cornwall", "East_Sussex", "Polperro", "Ashdown_Forest"
# ]

# wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

# wikivoyage_headers = {
#         "User-Agent": "building-llm-applications-ch08/1.0 student-notebook",
#         "Accept-Encoding": "gzip",
# }

# def load_wikivoyage_html(url):
#     try:
#         html_loader = AsyncHtmlLoader(
#             url,
#             header_template=wikivoyage_headers,
#             requests_per_second=0.2,
#             raise_for_status=True,
#             ignore_load_errors=False,
#         )
#         return html_loader.load()
#     except Exception as e:
#         print(f"Skipping {url}: {e}")
#         return []

In [8]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [9]:
print(uk_destination_urls)

['https://en.wikivoyage.org/wiki/Cornwall', 'https://en.wikivoyage.org/wiki/East_Sussex', 'https://en.wikivoyage.org/wiki/Polperro', 'https://en.wikivoyage.org/wiki/Ashdown_Forest']


In [ ]:
# for destination_url in uk_destination_urls:
#     html_loader = AsyncHtmlLoader(destination_url) #C
#     docs =  html_loader.load() #D

#     for doc in docs:
#         print(doc.metadata)
#         granular_chunks = split_docs_into_granular_chunks(docs)
#         uk_granular_collection.add_documents(documents=granular_chunks)

#         coarse_chunks = split_docs_into_coarse_chunks(docs)
#         uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists
#C Loader for one destination
#D Documents of one destination

In [ ]:
for destination_url in uk_destination_urls:
    docs = load_wikivoyage_html(destination_url) #C
    if not docs:
        continue

    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists
#C Loader for one destination
#D Documents of one destination

{'source': 'https://en.wikivoyage.org/wiki/Cornwall'}
{'source': 'https://en.wikivoyage.org/wiki/East_Sussex'}
{'source': 'https://en.wikivoyage.org/wiki/Polperro'}
{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest'}


#### Searching

In [ ]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

page_content='East Sussex' metadata={'Header 1': 'East Sussex'}
page_content='Sussex for free 
 [ edit ] 
 
 A Market during the Brighton Festival 
 There's plenty in Sussex for those who don't wish to spend plenty of cash on attractions: 
 
 
 Walking  - 3,500   km of walking paths, bridleways, scenic roads - all for free. 
 Go for a swim: Sussex has some of the cleanest beaches in the UK, with Brighton Beach renowned for its packed seafront, less well used areas, such as Eastbourne, Bexhill and Hastings still have facilities and cleanliness. 
 Brighton itself can be one big performance, the  Brighton Festival  and the  Brighton Festival Fringe , Features street performers, theatre groups, musicians, guided walks and a whole host of other great activities. 
 Town museums: Often they will charge, but some such as  Brighton Museum and Art Gallery  and Newhaven Museum are free (donations are gratefully welcomed though).' metadata={'Header 2': 'Sussex for free'}
page_content='Towns and vi

In [ ]:
print(granular_results)

[Document(id='12b88b1e-d7d0-4225-a52d-61cb5841a7cf', metadata={'Header 1': 'East Sussex'}, page_content='East Sussex'), Document(id='7d7fa2f6-b39b-4acc-8592-1f87530179c9', metadata={'Header 2': 'Sussex for free'}, page_content="Sussex for free \n [ edit ] \n \n A Market during the Brighton Festival \n There's plenty in Sussex for those who don't wish to spend plenty of cash on attractions: \n \n \n Walking  - 3,500 \xa0 km of walking paths, bridleways, scenic roads - all for free. \n Go for a swim: Sussex has some of the cleanest beaches in the UK, with Brighton Beach renowned for its packed seafront, less well used areas, such as Eastbourne, Bexhill and Hastings still have facilities and cleanliness. \n Brighton itself can be one big performance, the  Brighton Festival  and the  Brighton Festival Fringe , Features street performers, theatre groups, musicians, guided walks and a whole host of other great activities. \n Town museums: Often they will charge, but some such as  Brighton Mu

In [ ]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in Polperro",k=4)
for doc in coarse_results:
    print(doc)

page_content='[edit]

50°19′52″N 4°31′11″W

Map of Polperro

The walk into town from the parking lot is not very steep and takes 10
minutes. If walking is not your thing, there's a horse and cart or converted
milk float "tram" that will take you there and back for £1.50 (75p one way).

## See

[edit]

There are a couple of art galleries on the walk from the car park to the
harbour that may be worth a visit.The coastpath to the nearby town of Looe
also makes a pleasant walk on a summer's day.

  * 50.331685-4.5173881 Polperro Harbour Heritage Museum, 4 The Warren, PL13 2RB, ☏ +44 1503 272423. 10.30AM-4.30PM. £3 (adult). (updated Jul 2022)
  * 50.331648-4.5213152 Polperro Model Village, Mill Hill, PL13 2RP, ☏ +44 1503 272378. Miniature representation of the village, model railway and museum of local myths and legends. (updated Jul 2022)

## Do

[edit]

**Sea trips** from the harbour. There have been plenty of sightings of basking
sharks just off Polperro Harbour and the converted fishing

In [ ]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Regions 
 [ edit ] 
 
 .mw-parser-output .wv-staticMap{position:relative;left:-3px;margin-top:3px} 
 50°19′41″N 5°1′12″W Map of Cornwall 
 
 
Wikivoyage divides Cornwall into three regions. The  Isles of Scilly  are covered in a separate article. 
 
 .mw-parser-output .regionlistitem-table{vertical-align:middle;border-collapse:separate;border-spacing:2px;margin:0}.mw-parser-output .regionlistitem-table td{padding:0.15em 0.4em}.mw-parser-output .regionlistitem-textholder{transition:background-color 0.18s ease;border-radius:4px;cursor:pointer}.mw-parser-output .regionlistitem-textholder:hover{background-color:rgba(0,0,0,0.06);color:inherit}.mw-parser-output .regionlistitem-table:hover .regionlistitem-colorcell{background:inherit;color:inherit}@media screen{html.skin-theme-clientpref-night .mw-parser-output .regionlistitem-textholder:hover{background-color:rgba(255,255,255,0.06);color:inherit}}@media screen and (prefe

In [ ]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

page_content='_For other places with the same name, seeCornwall (disambiguation)._

**Cornwall** (Cornish: _Kernow_) is a county in the southwest of England.
Lying west of Devon from which it is separated by the River Tamar, Cornwall is
one of the more isolated and distinctive parts of the United Kingdom but is
also one of its most popular with holidaymakers. Its relatively warm climate,
long coastline, amazing scenery, and diverse Celtic heritage (combined with
tales of smuggling, pirates and King Arthur!) go only part of the way to
explaining its appeal.

The biomes that house the Eden Project, near St. Austell, Mid-Cornwall.

Cornwall is a popular destination for those interested in cultural tourism,
due to its long association with visual and written arts and its wealth of
archaeology. Its mining heritage has been recognised by the United Nations
(UNESCO). Over 30% of the county is designated as an Area of Outstanding
Natural Beauty (AONB), giving it national status and protection.

# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [27]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

In [28]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=embedding_model,
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [29]:
for destination_url in uk_destination_urls:
    html_docs = load_wikivoyage_html(destination_url) #A
    # html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Ingesting https://en.wikivoyage.org/wiki/Cornwall
Ingesting https://en.wikivoyage.org/wiki/East_Sussex
Ingesting https://en.wikivoyage.org/wiki/Polperro
Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [30]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['53d04b96-30ea-439c-8cfb-ed30c76d1483',
 'f3ecbaf9-1530-48f9-aa92-6577b631cd41',
 'fbd23db4-bc7e-4ac5-95ee-76082b483c41',
 '35bfbec6-29a0-497a-9406-43e9b1141226',
 '86f67ffc-5811-417a-8f24-a2fedb86cc87',
 '087e98ad-9004-4689-aaca-73babb567fbc',
 '58c8a091-c7ce-4065-96fc-cb31ae2336d5',
 'be5c9d1d-6247-46a7-a5f1-04dae3d3b1b8',
 '07dde433-3d24-46ee-a977-ce4f41947deb',
 '8faf0838-fecc-4e26-8376-cf21e2f8bf84',
 '652053eb-d494-40c0-aeab-b89f416f47ef',
 'ca2b614a-fa0e-4db1-a38c-0050ee8e3639',
 '232539cc-d410-4183-b128-e9b9b2b0cabd',
 '34b8c4f2-273e-42eb-8903-3ceb967a8908',
 '0b1a364a-5511-4ad7-b610-2503652ef0b5',
 '8ed751a8-5b72-4e9e-9c17-75d2a0cde133',
 '551b3d3b-afcc-4910-bafa-811cb7029c55',
 'ebeb3854-f005-49c2-90fc-0802de32a8ef',
 'c5bbf117-9bc8-458c-b1ea-db6d3ec9b02c',
 'a5edc0fa-32cd-4811-8165-edba3517caa5',
 '14c3decd-308e-4723-a29e-23d417ce1476',
 '2bf4fa31-1362-4023-9930-cc293d9db8f7',
 'f90e6cab-24e9-4ae1-8fc9-b71f23caaf74',
 'd7cc5fd5-30fb-48f5-8436-211400eeeaae',
 '99e56a09-4ce3-

### Performing a search on granular information

In [31]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [32]:
len(retrieved_docs)

3

In [33]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content="The Cornish have many significant saints. The pre-eminent patron saint is\nSaint Piran, whose flag, black with a white cross, is widely regarded as the\nnational emblem of Cornwall and can be seen all across the county. It is flown\nfrom private homes and government and public buildings. Saint Piran's Day is\nwidely celebrated on March 5 in Cornwall and amongst the Cornish diaspora\naround the globe.\n\nCornwall was a contributor to the Industrial Revolution, being famous\nparticularly for its copper and tin mining. Cornish miners have emigrated to\nmany parts of the world to the extent that the Cornish claim that a mine is\ndefined as being a hole in the ground with a Cornishman at its bottom. The\nCornish mines pioneered the use of stationary steam engines to power the\nmines. The Cornish are extremely proud of their history and heritage, which\npre-date the arrival of the Romans or Anglo-Saxons in

In [34]:
retrieved_docs[1]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='### Cornish\n\n[edit]\n\nBilingual welcome sign in Penzance station The Cornish word "tre" A common\nCornish word used in placenames is "tre" which refers to a farmstead, home,\nhomestead, town or village depending on context. Many towns and villages start\nwith this word such as: Trekenner, Treknow and Treburley  \n---  \n  \n**Cornish** (_Kernowek/Kernewek_) is a language belonging to the Brythonic\nbranch of the Celtic languages, and closely related to Breton and Welsh. It\nwas traditionally the dominant language of Cornwall, though the number of\nspeakers had diminished by the 17th century, and it became extinct some time\nlater. It is claimed that the last speaker was Dolly Pentreath, a fishwife\nfrom Mousehole, who passed away in Mousehole on 26 December 1777, although\nothers claim that there were Cornish-speakers who lived into the early 20th\ncentury.\n\nCornish was revived in the early 20th

### Comparing with direct semantic search on child chunks

In [35]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [36]:
len(child_docs_only)

4

In [37]:
child_docs_only[0]

Document(id='07fd990b-7385-4ce7-8207-0da8bf5f8dfd', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '35bfbec6-29a0-497a-9406-43e9b1141226'}, page_content='### Cornish\n\n[edit]')

In [ ]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

## Embedding child chunks with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [23]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=embedding_model,
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

### Ingesting the content into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    # html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs = load_wikivoyage_html(destination_url) #A
    # html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

Ingesting https://en.wikivoyage.org/wiki/Cornwall
Ingesting https://en.wikivoyage.org/wiki/East_Sussex
Ingesting https://en.wikivoyage.org/wiki/Polperro
Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

In [ ]:
len(retrieved_docs)

3

In [ ]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content="The Cornish have many significant saints. The pre-eminent patron saint is\nSaint Piran, whose flag, black with a white cross, is widely regarded as the\nnational emblem of Cornwall and can be seen all across the county. It is flown\nfrom private homes and government and public buildings. Saint Piran's Day is\nwidely celebrated on March 5 in Cornwall and amongst the Cornish diaspora\naround the globe.\n\nCornwall was a contributor to the Industrial Revolution, being famous\nparticularly for its copper and tin mining. Cornish miners have emigrated to\nmany parts of the world to the extent that the Cornish claim that a mine is\ndefined as being a hole in the ground with a Cornishman at its bottom. The\nCornish mines pioneered the use of stationary steam engines to power the\nmines. The Cornish are extremely proud of their history and heritage, which\npre-date the arrival of the Romans or Anglo-Saxons in

In [ ]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [ ]:
child_docs_only =  child_chunks_collection.similarity_search(
    "Cornwall Ranger")

In [ ]:
len(child_docs_only)

4

In [ ]:
child_docs_only[0]

Document(id='dde97b17-b3ff-4f10-befe-db31d82cde5d', metadata={'doc_id': 'f4a33fce-cd42-49c4-b1b6-30aa9b07b96a', 'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content='### Cornish\n\n[edit]')

In [ ]:
## IMPORTANT: Same as before

## Embedding summaries with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=embedding_model,
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the summarization chain

In [ ]:
# llm = ChatOpenAI(model="gpt-5-nano", openai_api_key=OPENAI_API_KEY)

In [ ]:
llm = ChatOpenAI(
    model="openai/gpt-5-nano",  # OpenRouter model slugs are prefixed, e.g. "openai/..."
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

In [ ]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    # html_loader = AsyncHtmlLoader(destination_url) #A
    # html_docs =  html_loader.load() #B
    html_docs = load_wikivoyage_html(destination_url) #A
    # html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text,
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

Ingesting https://en.wikivoyage.org/wiki/Cornwall
Ingesting https://en.wikivoyage.org/wiki/East_Sussex
Ingesting https://en.wikivoyage.org/wiki/Polperro
Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [ ]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

In [ ]:
len(retrieved_docs)

4

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall'}, page_content="**Go Cornwall Bus** buses operate between:\n\n  * **10** \\- Plymouth to Saltash, Looe and Polperro\n  * **11** \\- Plymouth to Saltash, Liskeard, Bodmin, Wadebridge and Padstow\n\n**Stagecoach** buses operate between Barnstaple, Holsworthy, Launceston and\nTavistock, across the Cornwall and Devon border (**85**).\n\n## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies (except certain town buses in St Ives and Fowey). The\n**Cornwall All Day ticket** allows unlimited travel for a calendar day. As of\n2025, day passes are £8 for adults and £5 for under-19s and £3 singles,\nregardless of age. Payment is by cash or contactless. Real time information\nand timetables can now be found through Transport for Cornwall (most reliable\nfor real time info), Transit (flaky & unreliable at times, and only buses) or

### Comparing with direct semantic search on summaries

In [ ]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [ ]:
len(summary_docs_only)

4

In [ ]:
summary_docs_only

[Document(id='d9f68df5-7445-4343-bb63-fc87788c4f0b', metadata={'doc_id': '0e92d49f-af03-4033-a804-68289f221b06'}, page_content='This is the Wikivoyage travel guide page for Cornwall, England. It serves as an outline index of topics you’d find in a full guide, organized as:\n\n- Regions; Towns and cities; Other destinations\n- Understand / Visitor information\n- Talk (English and Cornish)\n- Get in (By plane, By ferry, By train, By car, By coach)\n- Get around (By bus, By train, By ferry/boat)\n- See (National Trust properties, National Trust gardens)\n- Do\n- Eat (Savory, Sweet)\n- Drink (Ale & beer, Cider, Wine, Mead, Spirits)\n- Festivals\n- Sleep\n- Stay safe\n\nThe page also includes language options, related project links, and a note that a Cornwall disambiguation page exists. It is a navigational outline rather than filled-in content.'),
 Document(id='78e37afe-0cc0-49d1-8779-39d51e5492e5', metadata={'doc_id': '865a878b-5e9b-4230-9dd0-058b1e3ee05c'}, page_content='- Cornwall is a 

In [ ]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details.
# However, you might consider using the summaries directly if after testing they prove adequate.

## Embedding hypothetical questions with MultiVectorRetriever

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    # embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
    embedding_function=embedding_model,
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Setting up the chain to generate hypothetical questions

In [ ]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [ ]:
# llm_with_structured_output = ChatOpenAI(
#     model="gpt-5-nano",
#     openai_api_key=OPENAI_API_KEY).with_structured_output(
#         HypotheticalQuestions
# )

In [ ]:
llm_with_structured_output = ChatOpenAI(
    model="openai/gpt-5-nano",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
).with_structured_output(
    HypotheticalQuestions
)

In [ ]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

In [ ]:
for destination_url in uk_destination_urls:
    # html_loader = AsyncHtmlLoader(destination_url) #A
    # html_docs =  html_loader.load() #B
    html_docs = load_wikivoyage_html(destination_url) #A
    # html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E

        coarse_chunk_id = coarse_chunks_ids[i]

        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...nwall page available?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...nly city of Cornwall?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQues

Ingesting https://en.wikivoyage.org/wiki/Cornwall


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...isplayed on the page?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...ort to the continent?"]), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQues

Ingesting https://en.wikivoyage.org/wiki/East_Sussex


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...hat are their prices?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQuestions(que...use) can be expected?']), input_type=HypotheticalQuestions])
  return self.__pydantic_serializer__.to_python(
/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=HypotheticalQues

Ingesting https://en.wikivoyage.org/wiki/Polperro


APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 64482. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.', 'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 64482. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}]}}, 'user_id': 'user_3C7q8VGTZ7IKu48ON4baY8MOSsy'}

### Performing a search on granular information

In [ ]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

In [ ]:
len(retrieved_docs)

2

In [ ]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/East_Sussex'}, page_content='#### From the west\n\n[edit]\n\n  * **From Portsmouth**\n\nTrains come from Portsmouth Harbour and Portsmouth & Southsea stations to\nBrighton and Hove.\n\n  * **From Southampton**\n\nTrains come from Southampton Central to Brighton and Hove.\n\n#### From the east\n\n[edit]\n\n  * **From Ashford**\n\nTrains come from Ashford International to Hastings, Bexhill, Eastbourne and\nBrighton.\n\n#### From the north\n\n[edit]\n\n  * **From Bedford**\n\nTrains come from Bedford to Haywards Heath and Brighton, via St Pancras,\nLondon Blackfriars and Gatwick Airport.\n\n  * **From Reading**\n\nTrains come from Reading to Gatwick Airport, where you can change for trains\nto Brighton, Eastbourne, Hastings and other destinations.\n\n#### From the Continent\n\n[edit]\n\nTrains come from France and Europe through Calais and Ashford via the Eurostar\ntrain system. You will have to change trains at either St Pancr

### Inspecting possible questions matching our question through semantic search

In [ ]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [ ]:
len(hypothetical_question_docs_only)

4

In [ ]:
hypothetical_question_docs_only

[Document(id='3fa63b32-99f7-42ce-b539-3528d3701dd9', metadata={'doc_id': 'f5e08217-6fda-41a3-8657-a8517d5eaac8'}, page_content='If you are traveling from Portsmouth to Brighton, which departure stations could you use?'),
 Document(id='9de6b8f6-cc9c-4b09-9415-3f0ab1762fc8', metadata={'doc_id': 'f5e08217-6fda-41a3-8657-a8517d5eaac8'}, page_content='From Bedford, which destinations can you reach via Haywards Heath and Brighton, and what route would you take through major stations?'),
 Document(id='37138459-d52c-4806-ac64-50cd208a5719', metadata={'doc_id': 'a1d89aeb-8591-4f24-a812-b1a97a51ced9'}, page_content='Which bus operators serve East Sussex and Brighton & Hove, and what are some example routes or destinations mentioned (for example Brighton to Tunbridge Wells, Eastbourne to East Grinstead, and connections to Newhaven, Lewes, and Hastings)?'),
 Document(id='79520134-ed48-4d8d-9d2b-9fdc91502a4f', metadata={'doc_id': 'f5e08217-6fda-41a3-8657-a8517d5eaac8'}, page_content='If arriving fr

# Granular chunk expansion with MultiVectorRetriever

In [4]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [15]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    # embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY),
    embedding_function=embedding_model,
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

### Ingesting granular and expanded chunks into doc and vector store

In [6]:
# for destination_url in uk_destination_urls:
#     # html_loader = AsyncHtmlLoader(destination_url) #A
#     # html_docs =  html_loader.load() #B
#     text_docs = html2text_transformer.transform_documents(
#         html_docs) #C

#     granular_chunks = granular_chunk_splitter.split_documents(
#         text_docs) #D

#     expanded_chunk_store_items = []
#     for i, granular_chunk in enumerate(
#         granular_chunks): #E

#         this_chunk_num = i #F
#         previous_chunk_num = i-1 #F
#         next_chunk_num = i+1 #F

#         if i==0: #F
#             previous_chunk_num = None
#         elif i==(len(granular_chunks)-1): #F
#             next_chunk_num = None

#         expanded_chunk_text = "" #G
#         if previous_chunk_num: #G
#             expanded_chunk_text += granular_chunks[
#                 previous_chunk_num].page_content
#             expanded_chunk_text += "\n"

#         expanded_chunk_text += granular_chunks[
#             this_chunk_num].page_content #G
#         expanded_chunk_text += "\n"

#         if next_chunk_num: #G
#             expanded_chunk_text += granular_chunks[
#                 next_chunk_num].page_content
#             expanded_chunk_text += "\n"

#         expanded_chunk_id = str(uuid.uuid4()) #H
#         expanded_chunk_doc = Document(
#             page_content=expanded_chunk_text) #I

#         expanded_chunk_store_item = (expanded_chunk_id,
#                                      expanded_chunk_doc)
#         expanded_chunk_store_items.append(
#             expanded_chunk_store_item)

#         granular_chunk.metadata[
#             doc_key] = expanded_chunk_id #J

#     print(f'Ingesting {destination_url}')
#     multi_vector_retriever.vectorstore.add_documents(
#         granular_chunks) #K
#     multi_vector_retriever.docstore.mset(
#         expanded_chunk_store_items) #L

# #A Loader for one destination
# #B Documents of one destination
# #C transform HTML docs into clean text docs
# #D Split the destination content into granular chunks
# #E Iterate over the granular chunks
# #F determine the index of the current chunk and its previous and next chunks
# #G Assemble the text of the expanded chunk by including the previous and next chunk
# #H Generate the ID of the expanded chunk
# #I Create the expanded chunk document
# #J Link each granular chunk to its related expanded chunk
# #K Ingest the granular chunks into the vector store
# #L Ingest the expanded chunks into the document store

In [18]:
for destination_url in uk_destination_urls:
    # html_loader = AsyncHtmlLoader(destination_url) #A
    # html_docs =  html_loader.load() #B
    html_docs = load_wikivoyage_html(destination_url) #A
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F

        if i==0: #F
            previous_chunk_num = None
        if i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num is not None: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num is not None: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id,
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

Ingesting https://en.wikivoyage.org/wiki/Cornwall
Ingesting https://en.wikivoyage.org/wiki/East_Sussex
Ingesting https://en.wikivoyage.org/wiki/Polperro
Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [19]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

In [20]:
len(retrieved_docs)

4

In [21]:
retrieved_docs[0]

Document(metadata={}, page_content='### Visitor information\n\n[edit]\n\n  * Visit Cornwall (Tourist Board) website\n\n## Talk\n\n[edit]\n\nMost people in Cornwall speak English, while a very small minority also speak\nCornish. More people speak mainland European languages, notably Polish, than\nspeak Cornish.\n\n### English\n\n[edit]\n\nThe **English** dialect of Cornwall is distinctive; while to outsiders it\nsounds similar to other West Country accents, it is thought to have been\ninfluenced by Cornish in its phonology and intonation.\n### Cornish\n\n[edit]\nBilingual welcome sign in Penzance station The Cornish word "tre" A common\nCornish word used in placenames is "tre" which refers to a farmstead, home,\nhomestead, town or village depending on context. Many towns and villages start\nwith this word such as: Trekenner, Treknow and Treburley  \n---  \n  \n**Cornish** (_Kernowek/Kernewek_) is a language belonging to the Brythonic\nbranch of the Celtic languages, and closely related 

### Comparing with direct semantic search on granular chunks

In [38]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [39]:
len(child_docs_only)

4

In [40]:
child_docs_only[0]

Document(id='07fd990b-7385-4ce7-8207-0da8bf5f8dfd', metadata={'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'doc_id': '35bfbec6-29a0-497a-9406-43e9b1141226'}, page_content='### Cornish\n\n[edit]')

In [ ]:
# COMMENT: the expanded chunk has more useful context